In [3]:
"""
=============================================================================
  Ames House Price Prediction — Advanced Regression Ensemble
  Kaggle Competition: House Prices - Advanced Regression Techniques
  Author  : Kautsar Hilmi
  Metric  : RMSE on log(SalePrice)
=============================================================================
"""

# ─── 0. IMPORTS ──────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy import stats
from scipy.special import boxcox1p
from scipy.stats import boxcox_normmax

from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error

import lightgbm as lgb
import xgboost as xgb

# ─── 1. LOAD DATA ─────────────────────────────────────────────────────────────
print("=" * 60)
print("  STEP 1: LOADING DATA")
print("=" * 60)

train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")

print(f"  Train shape : {train.shape}")
print(f"  Test  shape : {test.shape}")

# Save IDs for final submission
test_id = test["Id"]

# ─── 2. EDA SUMMARY ───────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  STEP 2: EDA SUMMARY")
print("=" * 60)

print(f"\n  Target 'SalePrice' — mean: {train['SalePrice'].mean():,.0f}, "
      f"median: {train['SalePrice'].median():,.0f}, "
      f"skew: {train['SalePrice'].skew():.3f}")

# Missing values report
missing = train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(f"\n  Features with missing values (train): {len(missing)}")
print(missing.to_string())

# ─── 3. TARGET TRANSFORMATION (Log1p) ─────────────────────────────────────────
print("\n" + "=" * 60)
print("  STEP 3: TARGET TRANSFORMATION")
print("=" * 60)

# Log1p reduces skewness; inverse is expm1 at prediction time
y_train = np.log1p(train["SalePrice"])
print(f"  Skew before: {train['SalePrice'].skew():.3f}  → after log1p: {y_train.skew():.3f}")

# ─── 4. COMBINE TRAIN + TEST FOR CONSISTENT PROCESSING ───────────────────────
train_raw = train.drop(["Id", "SalePrice"], axis=1)
test_raw  = test.drop(["Id"], axis=1)
ntrain    = train_raw.shape[0]

all_data  = pd.concat([train_raw, test_raw], axis=0, ignore_index=True)
print(f"\n  Combined dataset shape: {all_data.shape}")

# ─── 5. MISSING VALUE IMPUTATION ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("  STEP 4: MISSING VALUE IMPUTATION")
print("=" * 60)

# ── 5a. Categorical columns where NA = "No Feature" (semantically meaningful)
none_cols = [
    "PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
    "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
    "MasVnrType"
]
for col in none_cols:
    all_data[col] = all_data[col].fillna("None")

# ── 5b. Numerical columns where NA = 0 (e.g., no basement ⟹ 0 sq ft)
zero_cols = [
    "GarageYrBlt", "GarageArea", "GarageCars",
    "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
    "BsmtFullBath", "BsmtHalfBath", "MasVnrArea"
]
for col in zero_cols:
    all_data[col] = all_data[col].fillna(0)

# ── 5c. LotFrontage: fill with median per Neighborhood
all_data["LotFrontage"] = all_data.groupby("Neighborhood")["LotFrontage"]\
                                   .transform(lambda x: x.fillna(x.median()))

# ── 5d. Remaining categoricals → mode
cat_mode_cols = ["MSZoning", "Electrical", "KitchenQual",
                 "Exterior1st", "Exterior2nd", "SaleType",
                 "Functional", "Utilities"]
for col in cat_mode_cols:
    all_data[col] = all_data[col].fillna(all_data[col].mode()[0])

print(f"  Remaining nulls after imputation: {all_data.isnull().sum().sum()}")

# ─── 6. FEATURE ENGINEERING ──────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  STEP 5: FEATURE ENGINEERING")
print("=" * 60)

# ── 6a. Area-based composite features
all_data["TotalSF"]        = (all_data["TotalBsmtSF"]
                               + all_data["1stFlrSF"]
                               + all_data["2ndFlrSF"])          # total living area

all_data["TotalBathrooms"] = (all_data["FullBath"]
                               + 0.5 * all_data["HalfBath"]
                               + all_data["BsmtFullBath"]
                               + 0.5 * all_data["BsmtHalfBath"])  # weighted bath count

all_data["TotalPorchSF"]   = (all_data["OpenPorchSF"]
                               + all_data["EnclosedPorch"]
                               + all_data["3SsnPorch"]
                               + all_data["ScreenPorch"])        # total porch area

# ── 6b. Age / time features
all_data["HouseAge"]       = all_data["YrSold"] - all_data["YearBuilt"]
all_data["RemodelAge"]     = all_data["YrSold"] - all_data["YearRemodAdd"]
all_data["IsRemodeled"]    = (all_data["YearBuilt"] != all_data["YearRemodAdd"]).astype(int)
all_data["IsNew"]          = (all_data["YrSold"] == all_data["YearBuilt"]).astype(int)

# ── 6c. Quality interaction features
all_data["OverallScore"]   = all_data["OverallQual"] * all_data["OverallCond"]
all_data["QualArea"]       = all_data["OverallQual"] * all_data["GrLivArea"]

# ── 6d. Has-feature binary flags
all_data["HasPool"]        = (all_data["PoolArea"] > 0).astype(int)
all_data["HasGarage"]      = (all_data["GarageArea"] > 0).astype(int)
all_data["HasFireplace"]   = (all_data["Fireplaces"] > 0).astype(int)
all_data["HasBasement"]    = (all_data["TotalBsmtSF"] > 0).astype(int)

print("  New features created: TotalSF, TotalBathrooms, TotalPorchSF,")
print("  HouseAge, RemodelAge, IsRemodeled, IsNew, OverallScore, QualArea,")
print("  HasPool, HasGarage, HasFireplace, HasBasement")

# ─── 7. SKEWNESS CORRECTION (Box-Cox on numerical features) ──────────────────
print("\n" + "=" * 60)
print("  STEP 6: SKEWNESS CORRECTION (Box-Cox)")
print("=" * 60)

numeric_feats  = all_data.dtypes[all_data.dtypes != "object"].index
skewed_feats   = all_data[numeric_feats].apply(lambda x: x.skew()).dropna()
skewed_feats   = skewed_feats[abs(skewed_feats) > 0.75].index

print(f"  Applying Box-Cox to {len(skewed_feats)} skewed features ...")
lam = 0.15  # common lambda for Box-Cox1p
for feat in skewed_feats:
    all_data[feat] = boxcox1p(all_data[feat], lam)

# ─── 8. ENCODE CATEGORICAL FEATURES ─────────────────────────────────────────
print("\n" + "=" * 60)
print("  STEP 7: ENCODING CATEGORICAL FEATURES")
print("=" * 60)

# Convert some ordinal-coded numerics that are actually categorical
all_data["MSSubClass"] = all_data["MSSubClass"].apply(str)
all_data["OverallCond"] = all_data["OverallCond"].astype(str)
all_data["YrSold"]     = all_data["YrSold"].astype(str)
all_data["MoSold"]     = all_data["MoSold"].astype(str)

all_data = pd.get_dummies(all_data)  # one-hot encoding
print(f"  Dataset shape after encoding: {all_data.shape}")

# ─── 9. SPLIT BACK INTO TRAIN / TEST ──────────────────────────────────────────
X_train = all_data[:ntrain]
X_test  = all_data[ntrain:]
print(f"\n  X_train: {X_train.shape}  |  X_test: {X_test.shape}")

# ─── 10. MODEL DEFINITIONS ───────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  STEP 8: MODEL DEFINITIONS")
print("=" * 60)

# Ridge with RobustScaler (handles outliers well)
ridge = make_pipeline(RobustScaler(), Ridge(alpha=10, random_state=42))

# Lasso with RobustScaler
lasso = make_pipeline(RobustScaler(), Lasso(alpha=0.0005, max_iter=10000, random_state=42))

# ElasticNet
enet  = make_pipeline(RobustScaler(), ElasticNet(alpha=0.0005, l1_ratio=0.9,
                                                  max_iter=10000, random_state=42))

# Gradient Boosting
gbr   = GradientBoostingRegressor(
    n_estimators=3000, learning_rate=0.05, max_depth=4,
    max_features="sqrt", min_samples_leaf=15, min_samples_split=10,
    loss="huber", random_state=42
)

# XGBoost
xgbr  = xgb.XGBRegressor(
    colsample_bytree=0.4603, gamma=0.0468, learning_rate=0.05,
    max_depth=3, min_child_weight=1.7817, n_estimators=2200,
    reg_alpha=0.4640, reg_lambda=0.8571, subsample=0.5213,
    silent=True, random_state=42, nthread=-1, tree_method="hist"
)

# LightGBM
lgbr  = lgb.LGBMRegressor(
    objective="regression", num_leaves=5, learning_rate=0.05,
    n_estimators=720, max_bin=55, bagging_fraction=0.8, bagging_freq=5,
    feature_fraction=0.2319, feature_fraction_seed=9, bagging_seed=9,
    min_child_samples=6, min_child_weight=0.001, min_split_gain=0.0,
    verbose=-1, random_state=42
)

models = {
    "Ridge"  : ridge,
    "Lasso"  : lasso,
    "ElasticNet" : enet,
    "GBR"    : gbr,
    "XGBoost": xgbr,
    "LightGBM": lgbr,
}

# ─── 11. K-FOLD CROSS-VALIDATION ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("  STEP 9: K-FOLD CROSS-VALIDATION (n=5)")
print("=" * 60)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

def rmsle_cv(model, X, y):
    """Return RMSLE scores across K folds (negative MSE → RMSE)."""
    scores = cross_val_score(
        model, X, y,
        scoring="neg_mean_squared_error",
        cv=kf, n_jobs=-1
    )
    return np.sqrt(-scores)

cv_scores = {}
for name, model in models.items():
    score = rmsle_cv(model, X_train, y_train)
    cv_scores[name] = score
    print(f"  {name:<12} RMSLE: {score.mean():.5f} ± {score.std():.5f}")

# ─── 12. TRAIN FULL DATASET & GENERATE OOF STACKING PREDICTIONS ─────────────
print("\n" + "=" * 60)
print("  STEP 10: TRAINING + STACKING ENSEMBLE")
print("=" * 60)

# Simple weighted average based on inverse CV error
weights = {name: 1 / cv_scores[name].mean() for name in models}
total_w = sum(weights.values())
weights = {name: w / total_w for name, w in weights.items()}

print("  Model weights (inverse RMSLE):")
for name, w in weights.items():
    print(f"    {name:<12}: {w:.4f}")

# Train each model on full training set
print("\n  Training all models on full data ...")
test_preds = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    test_preds[name] = model.predict(X_test)
    print(f"  ✓ {name} trained.")

# ─── 13. WEIGHTED ENSEMBLE PREDICTION ────────────────────────────────────────
print("\n" + "=" * 60)
print("  STEP 11: ENSEMBLE PREDICTION")
print("=" * 60)

ensemble_pred = np.zeros(X_test.shape[0])
for name in models:
    ensemble_pred += weights[name] * test_preds[name]

# Inverse log1p → original SalePrice scale
final_pred = np.expm1(ensemble_pred)
print(f"  Prediction stats — min: {final_pred.min():,.0f}  "
      f"max: {final_pred.max():,.0f}  mean: {final_pred.mean():,.0f}")

# ─── 14. GENERATE SUBMISSION FILE ────────────────────────────────────────────
print("\n" + "=" * 60)
print("  STEP 12: SAVING SUBMISSION FILE")
print("=" * 60)

submission = pd.DataFrame({
    "Id"       : test_id,
    "SalePrice": final_pred
})
submission.to_csv("submission.csv", index=False)
print("  ✅  submission.csv saved successfully!")
print(f"  Rows: {len(submission)}  |  Columns: {list(submission.columns)}")
print("\n" + "=" * 60)
print("  PIPELINE COMPLETE")
print("=" * 60)

  STEP 1: LOADING DATA
  Train shape : (1460, 81)
  Test  shape : (1459, 80)

  STEP 2: EDA SUMMARY

  Target 'SalePrice' — mean: 180,921, median: 163,000, skew: 1.883

  Features with missing values (train): 19
PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
BsmtFinType2      38
BsmtExposure      38
BsmtFinType1      37
BsmtCond          37
BsmtQual          37
MasVnrArea         8
Electrical         1

  STEP 3: TARGET TRANSFORMATION
  Skew before: 1.883  → after log1p: 0.121

  Combined dataset shape: (2919, 79)

  STEP 4: MISSING VALUE IMPUTATION
  Remaining nulls after imputation: 0

  STEP 5: FEATURE ENGINEERING
  New features created: TotalSF, TotalBathrooms, TotalPorchSF,
  HouseAge, RemodelAge, IsRemodeled, IsNew, OverallScore, QualArea,
  HasPool, HasGarage, HasFireplace, Has